# Run Xanthos-WM With Calibrated Parameters

This notebook is a step-by-step run guide for Xanthos-WM using existing calibrated parameters. It is written for a new user who has the code repository and the example data folder, but does not know the model internals.

The workflow uses `set_calibrate = -1`. That means the model loads `optimal_parameters.npy` and runs the calibrated model without estimating new parameters.

## What You Need

- The Xanthos-WM repository.
- The example data folder, referred to below as `/path/to/example`.
- A Python environment with the packages in `requirements.txt`.
- Write permission to `/path/to/example/output/`.

The example data folder should contain this structure:

```text
/path/to/example/
  input/
    calibration/
    climate/
    pet/
    reference/
    routing/
  output/
```

## Workflow

1. Create a Python environment.
2. Set the example-data path and basin number.
3. Validate required inputs.
4. Generate a local INI file for this machine.
5. Run `Basin.py` with calibrated parameters.
6. Check and inspect the output arrays.

## A. Run From Terminal

Use this path if you want to run the model without executing the notebook cells.

First create and activate the environment from the repository root:

```bash
cd /Users/abes208/Library/CloudStorage/OneDrive-PNNL/Documents/GitHub/xanthos-wm
conda create -n xanthos-wm python=3.12
conda activate xanthos-wm
python -m pip install -r requirements.txt
```

Then check `xanthos-wm/pm_abcd_mrtm_managed.ini`. At minimum, these values should match the machine and run you want:

```ini
RootDir = /path/to/example
n_basins = 1
basin_list = 218
Calibrate = 1
set_calibrate = -1
optimal_parameters = /Users/abes208/Library/CloudStorage/OneDrive-PNNL/Documents/GitHub/xanthos-wm/xanthos-wm/optimal_parameters.npy
```

Run the model from the inner model directory:

```bash
cd /Users/abes208/Library/CloudStorage/OneDrive-PNNL/Documents/GitHub/xanthos-wm/xanthos-wm
MPLCONFIGDIR=/tmp/matplotlib PYTHONPATH=. python Basin.py pm_abcd_mrtm_managed.ini
```

If you use this notebook to generate `pm_abcd_mrtm_managed.local.ini`, run the generated file instead:

```bash
MPLCONFIGDIR=/tmp/matplotlib PYTHONPATH=. python Basin.py pm_abcd_mrtm_managed.local.ini
```

Successful calibrated runs write outputs to `/path/to/example/output/calibration/`.

## B. Run From This Notebook

The notebook path below generates a local INI file, runs the model, and checks the outputs. This avoids manually editing the tracked template INI.

## 1. Set Run Options

Edit `EXAMPLE_DIR` for the machine running the model. Use `/Users/abes208/Downloads/example` on this workstation, or replace it with the actual example-data path on another machine.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

import numpy as np
import pandas as pd
from configobj import ConfigObj

# EDIT THIS PATH if the example data are somewhere else.
EXAMPLE_DIR = Path('/Users/abes208/Downloads/example')

# Active calibrated example basin.
BASIN_ID = 218
START_YEAR = 1971
END_YEAR = 2001

# Leave this as None unless auto-detection cannot find the repository root.
REPO_DIR = None


def find_repo_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in [start, *start.parents]:
        has_requirements = (candidate / 'requirements.txt').exists()
        has_model_ini = (candidate / 'xanthos-wm' / 'pm_abcd_mrtm_managed.ini').exists()
        if has_requirements and has_model_ini:
            return candidate
    raise FileNotFoundError('Could not find the repository root. Set REPO_DIR manually.')


REPO_DIR = Path(REPO_DIR).resolve() if REPO_DIR else find_repo_root()
MODEL_DIR = REPO_DIR / 'xanthos-wm'
TEMPLATE_INI = MODEL_DIR / 'pm_abcd_mrtm_managed.ini'
LOCAL_INI = MODEL_DIR / 'pm_abcd_mrtm_managed.local.ini'
OPTIMAL_PARAMETERS = MODEL_DIR / 'optimal_parameters.npy'

print(f'Repository:      {REPO_DIR}')
print(f'Model directory: {MODEL_DIR}')
print(f'Example data:    {EXAMPLE_DIR}')
print(f'Basin ID:        {BASIN_ID}')

## 2. Validate Required Inputs

Run this before launching the model. It catches missing paths, wrong example-data locations, and basin IDs that are not present in the calibrated-parameter file.

In [ ]:
required_paths = {
    'example directory': EXAMPLE_DIR,
    'template INI': TEMPLATE_INI,
    'optimal parameters': OPTIMAL_PARAMETERS,
    'PET input': EXAMPLE_DIR / 'input/pet/penman_monteith/penman_monteith_watch_monthly_pet_1971_2001.npy',
    'minimum temperature': EXAMPLE_DIR / 'input/climate/tasmin_watch_monthly_degc_1971_2001.npy',
    'precipitation': EXAMPLE_DIR / 'input/climate/pr_gpcc_watch_monthly_mmpermth_1971_2001.npy',
    'GRDC stations': EXAMPLE_DIR / 'input/GRDC_stations_selected_final_list_2.csv',
    'GRDC streamflow': EXAMPLE_DIR / 'input/calibration/grdc_95basin_m3persec_1971_1990_monthly.csv',
    'reservoir NetCDF': EXAMPLE_DIR / 'input/calibration/Xanthos_reservoirs_halfDegree_global.nc',
    'routing flow distance': EXAMPLE_DIR / 'input/routing/mrtm/DRT_half_FDISTANCE_globe.txt',
    'routing flow direction': EXAMPLE_DIR / 'input/routing/mrtm/DRT_half_FDR_globe_bystr50.txt',
    'routing velocity': EXAMPLE_DIR / 'input/routing/mrtm/velocity_half_degree.npy',
}

missing = []
for label, path in required_paths.items():
    ok = path.exists()
    print(f'{label:24s}: {path}  {"OK" if ok else "MISSING"}')
    if not ok:
        missing.append((label, path))

if missing:
    details = '\n'.join(f'- {label}: {path}' for label, path in missing)
    raise FileNotFoundError(f'Missing required inputs. Update EXAMPLE_DIR or restore the data bundle.\n{details}')

params = np.load(OPTIMAL_PARAMETERS)
if params.ndim != 2 or params.shape[1] <= 7:
    raise ValueError(f'Unexpected optimal parameter shape: {params.shape}')
if not np.any(params[:, 7] == BASIN_ID):
    raise ValueError(f'Basin {BASIN_ID} was not found in {OPTIMAL_PARAMETERS}')

stations = pd.read_csv(required_paths['GRDC stations'])
if 'basins' in stations.columns and BASIN_ID in set(stations['basins'].astype(int)):
    display(stations.loc[stations['basins'].astype(int) == BASIN_ID])
else:
    print(f'Warning: basin {BASIN_ID} was not found in the GRDC station metadata.')

## 3. Generate A Local INI File

This writes `pm_abcd_mrtm_managed.local.ini` beside the template INI. The generated file is machine-specific and can be recreated whenever paths or basin choices change.

In [ ]:
cfg = ConfigObj(str(TEMPLATE_INI), list_values=False)

cfg['Project']['RootDir'] = str(EXAMPLE_DIR)
cfg['Project']['InputFolder'] = 'input'
cfg['Project']['OutputFolder'] = 'output'
cfg['Project']['n_basins'] = '1'
cfg['Project']['basin_list'] = str(BASIN_ID)
cfg['Project']['StartYear'] = str(START_YEAR)
cfg['Project']['EndYear'] = str(END_YEAR)
cfg['Project']['Calibrate'] = '1'

cfg['PET']['pet_module'] = 'none'
cfg['PET']['pet_file'] = str(EXAMPLE_DIR / 'input/pet/penman_monteith/penman_monteith_watch_monthly_pet_1971_2001.npy')

cfg['Runoff']['runoff_module'] = 'abcd_managed'
cfg['Runoff']['abcd_managed']['runoff_spinup'] = '372'
cfg['Runoff']['abcd_managed']['TempMinFile'] = str(EXAMPLE_DIR / 'input/climate/tasmin_watch_monthly_degc_1971_2001.npy')
cfg['Runoff']['abcd_managed']['PrecipitationFile'] = str(EXAMPLE_DIR / 'input/climate/pr_gpcc_watch_monthly_mmpermth_1971_2001.npy')

cfg['Routing']['routing_module'] = 'mrtm_managed'
cfg['Routing']['mrtm_managed']['routing_spinup'] = '372'
cfg['Routing']['mrtm_managed']['grdc_coord_index_file'] = str(EXAMPLE_DIR / 'input/GRDC_stations_selected_final_list_2.csv')
cfg['Routing']['mrtm_managed']['Xanthos_wm_file'] = str(EXAMPLE_DIR / 'input/calibration/Xanthos_reservoirs_halfDegree_global.nc')
cfg['Routing']['mrtm_managed']['optimal_parameters'] = str(OPTIMAL_PARAMETERS)

cfg['Calibrate']['set_calibrate'] = '-1'
cfg['Calibrate']['observed'] = str(EXAMPLE_DIR / 'input/calibration/grdc_95basin_m3persec_1971_1990_monthly.csv')
cfg['Calibrate']['obs_unit'] = 'm3_per_sec'
cfg['Calibrate']['calib_out_dir'] = str(EXAMPLE_DIR / 'output/calibration')
cfg['Calibrate']['repetitions'] = '1'

cfg.filename = str(LOCAL_INI)
cfg.write()

print(f'Wrote local configuration: {LOCAL_INI}')

## 4. Terminal Command For This Local INI

This cell prints the exact terminal command for the local INI generated above. Use it if you want to run outside Jupyter.

In [ ]:
print(f'cd {MODEL_DIR}')
print(f'MPLCONFIGDIR=/tmp/matplotlib PYTHONPATH=. python Basin.py {LOCAL_INI.name}')

## 5. Run The Model From Jupyter

This runs `Basin.py` with the generated local INI. The run can take a few minutes and may be quiet during routing.

In [ ]:
env = os.environ.copy()
env.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')
env['PYTHONPATH'] = str(MODEL_DIR)

cmd = [sys.executable, 'Basin.py', str(LOCAL_INI)]
print('Running:', ' '.join(cmd))

result = subprocess.run(
    cmd,
    cwd=MODEL_DIR,
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(result.stdout)
result.check_returncode()

## 6. Confirm Output Files

A successful calibrated run writes runoff and managed streamflow arrays under the calibration output folder.

In [ ]:
output_dir = EXAMPLE_DIR / 'output/calibration'
runoff_output = output_dir / f'Simulated_Runoff_mm_per_month{BASIN_ID}.npy'

# The print statement in the model omits underscores, but the file writer uses this name.
streamflow_candidates = [
    output_dir / f'Simulated_Streamflow_m3_per_sec{BASIN_ID}.npy',
    output_dir / f'Simulated_Streamflow_m3persec{BASIN_ID}.npy',
]
streamflow_output = next((path for path in streamflow_candidates if path.exists()), streamflow_candidates[0])

for label, path in [('runoff', runoff_output), ('streamflow', streamflow_output)]:
    print(f'{label:10s}: {path}  {"OK" if path.exists() else "MISSING"}')
    if not path.exists():
        raise FileNotFoundError(path)

## 7. Inspect Results

This is a basic sanity check of array shape, missing values, and value ranges. It is not a formal validation.

In [ ]:
runoff = np.load(runoff_output)
streamflow = np.load(streamflow_output)


def summarize(name, array):
    return {
        'name': name,
        'shape': array.shape,
        'nan_count': int(np.isnan(array).sum()),
        'min': float(np.nanmin(array)),
        'max': float(np.nanmax(array)),
        'mean': float(np.nanmean(array)),
    }


pd.DataFrame([
    summarize('runoff_mm_per_month', runoff),
    summarize('streamflow_m3_per_sec', streamflow),
])

## 8. Optional Plot

This plot gives a quick visual check of the basin-average managed streamflow.

In [ ]:
import matplotlib.pyplot as plt

basin_mean_streamflow = np.nanmean(streamflow, axis=0)
dates = pd.period_range(f'{START_YEAR}-01', periods=basin_mean_streamflow.size, freq='M').to_timestamp()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(dates, basin_mean_streamflow, linewidth=1.2)
ax.set_title(f'Basin {BASIN_ID} mean simulated managed streamflow')
ax.set_ylabel('m3/s')
ax.set_xlabel('Date')
ax.grid(True, alpha=0.3)
plt.show()

## Troubleshooting

- If validation reports missing files, update `EXAMPLE_DIR` or restore the example data folder.
- If `configobj` is missing, run `python -m pip install -r requirements.txt` from the repository root.
- If the model cannot write logs or outputs, check write permission under `/path/to/example/output/`.
- Warnings about NaNs in the precipitation or temperature inputs are expected for this data bundle.
- `set_calibrate = -1` is the calibrated-parameter run mode. `set_calibrate = 0` or `1` starts calibration and takes much longer.